# Solution 03 — batch producer & partitions

In [ ]:
from confluent_kafka import Producer
from collections import defaultdict, Counter
import json, time, random

producer = Producer({'bootstrap.servers': 'redpanda:29092', 'client.id': 'batch-producer'})
houses = ['haus_a', 'haus_b', 'haus_c']
topics = ['strom', 'wasser']

In [ ]:
results = []
def delivery_report(err, msg):
    if not err:
        results.append({'topic': msg.topic(), 'partition': msg.partition(),
                        'key': msg.key().decode()})

for _ in range(15):
    house = random.choice(houses)
    topic = random.choice(topics)
    value = round(random.uniform(1.0, 100.0), 2)
    event = json.dumps({'sensor': topic, 'haus': house, 'wert': value,
                        'einheit': 'kWh' if topic == 'strom' else 'Liter',
                        'timestamp': time.time()})
    producer.produce(topic, key=house.encode(), value=event.encode(),
                     callback=delivery_report)
producer.flush()

key_partitions = defaultdict(set)
for r in results:
    key_partitions[(r['topic'], r['key'])].add(r['partition'])
for (t, k), parts in sorted(key_partitions.items()):
    print(f'{t:<7} | {k:<7} | {sorted(parts)}')

## Task A — keyless events

Notice the distribution is *not* uniform — librdkafka uses 'sticky' partitioning since 1.4 for better batching. Across many runs the partitions average out, but in a 6-event burst you usually see one partition dominate.

In [ ]:
no_key_partitions = []
def no_key_cb(err, msg):
    if not err:
        no_key_partitions.append(msg.partition())
        print(f'  -> P{msg.partition()} offset={msg.offset()}')

for i in range(6):
    event = json.dumps({'sensor': 'strom', 'wert': float(i),
                        'einheit': 'kWh', 'timestamp': time.time()})
    producer.produce('strom', key=None, value=event.encode(), callback=no_key_cb)

producer.flush()
print(f'Distribution: {dict(Counter(no_key_partitions))}')